In [ ]:
import time
import itertools
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pennylane as qml
from scipy.optimize import minimize

# =========================
# 1. Generate random graph
# =========================

seed = 13202050   # TODO: replace with your student ID
n = 8

G = nx.gnp_random_graph(n=n, p=0.5, seed=seed)
edges = list(G.edges())

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Edge list:", edges)

plt.figure(figsize=(5, 4))
pos = nx.spring_layout(G, seed=seed)
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=700)
nx.draw_networkx_edges(G, pos)
plt.title("Random Graph for Max-Cut")
plt.show()


# =========================
# 2. Brute-force Max-Cut
# =========================

def cut_value(bitstring, edges):
    value = 0
    for i, j in edges:
        if bitstring[i] != bitstring[j]:
            value += 1
    return value

best_cut = -1
optimal_partitions = []

start = time.time()

for bits in itertools.product([0, 1], repeat=n):
    value = cut_value(bits, edges)
    if value > best_cut:
        best_cut = value
        optimal_partitions = [bits]
    elif value == best_cut:
        optimal_partitions.append(bits)

bruteforce_time = time.time() - start

print("\n=== Brute Force Result ===")
print("Maximum cut value:", best_cut)
print("Number of optimal partitions:", len(optimal_partitions))
print("Optimal partitions:")
for ptn in optimal_partitions:
    print(ptn)
print("Brute-force time:", bruteforce_time)


# =========================
# 3. PennyLane QAOA setup
# =========================

dev = qml.device("default.qubit", wires=n)

def qaoa_circuit(params, p):
    gammas = params[:p]
    betas = params[p:]

    for wire in range(n):
        qml.Hadamard(wires=wire)

    for layer in range(p):
        gamma = gammas[layer]
        beta = betas[layer]

        # Cost unitary for Max-Cut
        for i, j in edges:
            qml.CNOT(wires=[i, j])
            qml.RZ(-gamma, wires=j)
            qml.CNOT(wires=[i, j])

        # Mixer unitary
        for wire in range(n):
            qml.RX(2 * beta, wires=wire)

def hamiltonian_terms():
    coeffs = []
    obs = []

    for i, j in edges:
        coeffs.append(-0.5)
        obs.append(qml.PauliZ(i) @ qml.PauliZ(j))

    coeffs.append(len(edges) / 2)
    obs.append(qml.Identity(0))

    return qml.Hamiltonian(coeffs, obs)

H_C = hamiltonian_terms()

@qml.qnode(dev)
def energy_qnode(params, p):
    qaoa_circuit(params, p)
    return qml.expval(H_C)

@qml.qnode(dev)
def probability_qnode(params, p):
    qaoa_circuit(params, p)
    return qml.probs(wires=range(n))


# =========================
# 4. Energy landscape for p = 1
# =========================

gamma_vals = np.linspace(0, 2 * np.pi, 80)
beta_vals = np.linspace(0, np.pi, 80)

landscape = np.zeros((len(beta_vals), len(gamma_vals)))

for b_idx, beta in enumerate(beta_vals):
    for g_idx, gamma in enumerate(gamma_vals):
        params = np.array([gamma, beta])
        landscape[b_idx, g_idx] = energy_qnode(params, p=1)

min_idx = np.unravel_index(np.argmin(landscape), landscape.shape)
global_beta = beta_vals[min_idx[0]]
global_gamma = gamma_vals[min_idx[1]]
global_energy = landscape[min_idx]

plt.figure(figsize=(7, 5))
plt.imshow(
    landscape,
    extent=[0, 2*np.pi, 0, np.pi],
    origin="lower",
    aspect="auto"
)
plt.colorbar(label=r"$F(\gamma,\beta)$")
plt.scatter(global_gamma, global_beta, marker="x", s=100, color="red")
plt.xlabel(r"$\gamma$")
plt.ylabel(r"$\beta$")
plt.title("QAOA p=1 Energy Landscape")
plt.show()

print("\n=== p = 1 Energy Landscape ===")
print("Global minimum gamma:", global_gamma)
print("Global minimum beta:", global_beta)
print("Minimum energy:", global_energy)


# =========================
# 5. Optimize QAOA for p = 1,2,3,4
# =========================

def most_likely_bitstring(params, p):
    probs = probability_qnode(params, p)
    index = np.argmax(probs)
    bits = tuple(int(x) for x in format(index, f"0{n}b"))
    return bits, probs[index]

qaoa_results = []

for p in [1, 2, 3, 4]:
    best_result = None

    start = time.time()

    # multiple random starts to reduce local minimum problem
    for trial in range(10):
        init_gammas = np.random.uniform(0, 2*np.pi, p)
        init_betas = np.random.uniform(0, np.pi, p)
        init_params = np.concatenate([init_gammas, init_betas])

        result = minimize(
            lambda x: energy_qnode(x, p),
            init_params,
            method="COBYLA",
            options={"maxiter": 200}
        )

        if best_result is None or result.fun < best_result.fun:
            best_result = result

    qaoa_time = time.time() - start

    opt_params = best_result.x
    bitstring, probability = most_likely_bitstring(opt_params, p)
    found_cut = cut_value(bitstring, edges)
    ratio = found_cut / best_cut

    qaoa_results.append({
        "method": f"QAOA p={p}",
        "best_cut": found_cut,
        "approx_ratio": ratio,
        "time": qaoa_time,
        "params": opt_params,
        "bitstring": bitstring,
        "probability": probability
    })

    print(f"\n=== QAOA p={p} ===")
    print("Optimized parameters:", opt_params)
    print("Most likely bitstring:", bitstring)
    print("Probability:", probability)
    print("Best cut found:", found_cut)
    print("Approximation ratio:", ratio)
    print("Computation time:", qaoa_time)


# =========================
# 6. Simulated Annealing
# =========================

try:
    import dimod
    from neal import SimulatedAnnealingSampler

    # Max-Cut as QUBO: minimize -cut
    Q = {}

    for i, j in edges:
        Q[(i, i)] = Q.get((i, i), 0) - 1
        Q[(j, j)] = Q.get((j, j), 0) - 1
        Q[(i, j)] = Q.get((i, j), 0) + 2

    bqm = dimod.BinaryQuadraticModel.from_qubo(Q)

    start = time.time()
    sampler = SimulatedAnnealingSampler()
    sampleset = sampler.sample(bqm, num_reads=1000)
    sa_time = time.time() - start

    best_sample = sampleset.first.sample
    sa_bits = tuple(best_sample[i] for i in range(n))
    sa_cut = cut_value(sa_bits, edges)
    sa_ratio = sa_cut / best_cut

except ImportError:
    print("\nPlease install simulated annealing packages:")
    print("pip install dimod dwave-neal")
    sa_bits = None
    sa_cut = None
    sa_ratio = None
    sa_time = None


# =========================
# 7. Final comparison table
# =========================

comparison = []

comparison.append({
    "method": "Brute force",
    "best_cut": best_cut,
    "approx_ratio": 1.0,
    "time": bruteforce_time
})

if sa_cut is not None:
    comparison.append({
        "method": "Simulated annealing",
        "best_cut": sa_cut,
        "approx_ratio": sa_ratio,
        "time": sa_time
    })

for r in qaoa_results:
    comparison.append({
        "method": r["method"],
        "best_cut": r["best_cut"],
        "approx_ratio": r["approx_ratio"],
        "time": r["time"]
    })

print("\n=== Comparison Table ===")
print(f"{'Method':<25} {'Best Cut':<10} {'Approx. Ratio':<15} {'Time (s)':<10}")
print("-" * 65)

for row in comparison:
    print(
        f"{row['method']:<25} "
        f"{row['best_cut']:<10} "
        f"{row['approx_ratio']:<15.4f} "
        f"{row['time']:<10.4f}"
    )